# GX_01 — Loading and Checking the Data

**The question this notebook answers:** *Can we trust the data before we use it?*

### What this notebook is about
Our client, a bank, gave us two files. Before anyone analyses them or builds a model, we have to be sure the data is complete and correct. If the data is broken, every result that comes after it is wrong, no matter how good the model is.

In this notebook we:
1. **Load** both files into Spark.
2. **Save** them as tables that the rest of the project will use.
3. **Check** them for problems: missing values, duplicates, impossible values.
4. **Write down** what we found, so the next notebook knows what to clean.

> **Important rule:** here we only *look* for problems. We do not fix or delete anything. Fixing happens in the next notebook (GX_02), where every change is a deliberate and documented decision.

### The two files
| File | What it contains | Size |
|---|---|---|
| `churn.csv` | One row per bank customer: age, country, balance, number of products, and whether they left the bank (`Exited`) | 10,000 customers |
| `bank_transactions.csv` | One row per money transfer between two accounts | 1,000 transactions |

### What this notebook produces
| Table | What is in it |
|---|---|
| `bronze_churn` | The customer file saved as a table, values untouched |
| `bronze_transactions` | The transaction file saved as a table, values untouched |
| `quality_report` | Every check we ran and its result |

### Why do we call it "bronze"?
Big data projects usually store data in three layers:
- **Bronze** — an untouched copy of the raw data
- **Silver** — cleaned data
- **Gold** — data ready for analysis and models

Keeping an untouched copy means that if we ever make a mistake while cleaning, we can simply redo the cleaning from the original, without going back to the files.

### Why save tables instead of just reading the CSV each time?
A CSV is plain text: Spark has to re-read and re-interpret it every time. A **Delta table** stores the data in an efficient format, with fixed column types and a history of every change, and every notebook can read it quickly. With 10,000 rows the difference is small, but for a real bank with millions of customers it matters a lot.

## Step 0 — Preparing the notebook

**What we do:** load the shared helper code our team wrote (the `src/` folder) and check that Spark is running.

**Why:** instead of copying the same code into every notebook, we keep it in one place:
- `config.py` — where files and tables are stored
- `schemas.py` — the exact columns and types we expect in each file
- `ingest.py` — how to load a file and save it as a table
- `quality.py` — the data quality checks

If something changes (for example a folder name), we change it once and every notebook follows.

In [0]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from pyspark.sql import functions as F

from src.config import (
    BRONZE_CHURN, BRONZE_TRANSACTIONS, CHURN_CSV, EXPECTED_ROWS,
    QUALITY_REPORT, RAW_PATH, TRANSACTIONS_CSV,
)
from src.ingest import META_COLS, read_raw_csv, write_bronze
from src.quality import category_counts, quality_report
from src.schemas import CHURN_SCHEMA, CORRUPT_COL, TRANSACTIONS_SCHEMA

print("Spark version:", spark.version)

Spark version: 4.2.0


**Result:** Spark 4.2.0 is running and the helper code loaded without errors. We are ready to start.

**What we do:** list the files in our storage folder (a Databricks *Volume*) to confirm that both files are there.

In [0]:
# Raw files in the Volume (uploaded once in GX_00_setup)
display(dbutils.fs.ls(RAW_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/churn/raw/bank_transactions.csv,bank_transactions.csv,131709,1793352640000
dbfs:/Volumes/workspace/churn/raw/churn.csv,churn.csv,684858,1793352644000


**Result:** both files are there, `churn.csv` (about 685 KB) and `bank_transactions.csv` (about 132 KB). These are small files. The point of the project is to show the client that the same process also works on files with millions of rows.

## Step 1 — Customer data (`churn.csv`)

### 1.1 Loading the file

**What we do:** read the customer file and save it as the table `bronze_churn`.

**How we read it, and why:**
- **We tell Spark the exact type of every column** (whole number, decimal, text) instead of letting Spark guess. Guessing means reading the file twice, and it can go wrong without anyone noticing: one strange value could turn a number column into text. With fixed types, anything that does not fit gets caught.
- **We keep broken rows instead of dropping them.** If a row does not match the expected format, Spark puts it into a special column called `_corrupt_record`. That way we can count problems instead of silently losing data.
- **We add two tracking columns:** `_ingested_at` (when we loaded the data) and `_source_file` (which file it came from). If a number ever looks wrong later, we can trace it back to its source.
- **Text encoding:** we read the file as ISO-8859-1, a standard for European characters. Note that 92 surnames contain a `?` (for example "H?"). We checked the original file: the `?` is already there, so the real letters were lost before we received the data. Surnames are never used in the analysis, so this has no effect.

In [0]:
raw_churn = read_raw_csv(spark, CHURN_CSV, CHURN_SCHEMA, encoding="ISO-8859-1")
write_bronze(raw_churn, BRONZE_CHURN)

bronze_churn = spark.table(BRONZE_CHURN)
print(f"{BRONZE_CHURN}: {bronze_churn.count():,} rows, {len(bronze_churn.columns)} columns")
bronze_churn.printSchema()

workspace.churn.bronze_churn: 10,000 rows, 17 columns
root
 |-- RowNumber: integer (nullable = true)
 |-- CustomerId: long (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: integer (nullable = true)
 |-- IsActiveMember: integer (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)
 |-- _corrupt_record: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



**Result:** **10,000 rows**, exactly the number we expected, so nothing was lost or duplicated while loading.

There are **17 columns**: the 14 original columns plus the three we added (`_corrupt_record`, `_ingested_at`, `_source_file`). Every column has the type we intended. For example, `Age` is a whole number and `Balance` is a decimal number.

**What we do:** look at the first 5 customers to check that the values make sense. We hide `Surname` because it is personal information.

In [0]:
# Preview without Surname (personal data is not displayed)
display(bronze_churn.drop("Surname", *META_COLS).limit(5))

RowNumber,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,_corrupt_record
1,15634602,619,France,Female,42,2,0.0,1,1,1,101348.88,1,null
2,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,null
3,15619304,502,France,Female,42,8,159660.8,3,1,0,113931.57,1,null
4,15701354,699,France,Female,39,1,0.0,2,0,0,93826.63,0,null
5,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0,null


**Result:** the values look right. For example, customer 1 is a 42-year-old woman from France with a balance of 0 who left the bank (`Exited = 1`).

The `_corrupt_record` column is empty (`null`) for every row. That is good news: it means these rows were read without any problem.

### 1.2 Quality checks

**What we do:** run our reusable quality function on the customer table. It answers five questions:
- Is the number of rows what we expected?
- Are there missing values (empty cells)?
- Are there duplicate rows or duplicate customer IDs?
- Are the numbers inside a sensible range (for example, nobody aged 150)?
- Do text columns contain only expected values (for example, Geography is only France, Germany or Spain)?

Every check gets a status:

| Status | Meaning |
|---|---|
| **PASS** | All good |
| **WARN** | Worth a look, but not serious |
| **FAIL** | Serious problem, must be fixed |
| **INFO** | Not a check, just a useful number |

**The limits we use** are common-sense limits, not business rules:
- Credit score between 300 and 850 (the usual credit score scale)
- Age between 18 and 100 (adult customers)
- Tenure between 0 and 10 years (as described in the dataset documentation)
- Number of products between 1 and 4
- Balance and salary not negative

**Why this matters:** a model trained on bad data gives bad answers. These checks take a few seconds and protect everything that comes after.

In [0]:
churn_quality = quality_report(
    bronze_churn,
    dataset="churn",
    expected_rows=EXPECTED_ROWS["churn"],
    key_col="CustomerId",
    ranges={
        "CreditScore": (300, 850),
        "Age": (18, 100),
        "Tenure": (0, 10),
        "NumOfProducts": (1, 4),
        "Balance": (0, 10_000_000),
        "EstimatedSalary": (0, 10_000_000),
    },
    allowed_values={
        "Geography": ["France", "Germany", "Spain"],
        "Gender": ["Male", "Female"],
        "HasCrCard": [0, 1],
        "IsActiveMember": [0, 1],
        "Exited": [0, 1],
    },
    rate_cols=["Exited", "HasCrCard", "IsActiveMember"],
)

# Summary first, then only the rows that need attention (full report is saved in section 3)
display(churn_quality.groupBy("status").count().orderBy("status"))
display(churn_quality.filter("status != 'PASS'"))

status,count
INFO,3
PASS,34


dataset,check,column,observed,expected,status
churn,positive_rate,Exited,0.2037,-,INFO
churn,positive_rate,HasCrCard,0.7055,-,INFO
churn,positive_rate,IsActiveMember,0.5151,-,INFO


**Result:** **34 checks passed, 0 warnings, 0 failures.** The customer data is complete and clean: no missing values, no duplicates, no impossible values.

The 3 INFO lines are useful numbers, not problems:
- **20.4% of customers left the bank** (`Exited`). This is what we are trying to predict. Only 1 in 5 customers leaves, so the two groups are uneven. We have to keep that in mind when we judge our model later: a model that always says "stays" would already be right about 80% of the time, without being useful at all.
- **70.6%** of customers have a credit card.
- **51.5%** are active members.

**What we do:** count how many customers fall into each category. This confirms there are no spelling mistakes (like "france" next to "France") and shows how big each group is.

In [0]:
# Category distributions: confirms no typos or unexpected labels
display(category_counts(bronze_churn, ["Geography", "Gender", "NumOfProducts", "HasCrCard", "IsActiveMember", "Exited"]))

column,value,n,pct
Exited,0,7963,79.63
Exited,1,2037,20.37
Gender,Male,5457,54.57
Gender,Female,4543,45.43
Geography,France,5014,50.14
Geography,Germany,2509,25.09
Geography,Spain,2477,24.77
HasCrCard,1,7055,70.55
HasCrCard,0,2945,29.45
IsActiveMember,1,5151,51.51


**Result:** all categories are clean, with no unexpected values. Some useful facts:
- **Country:** half of the customers are in France (50%), and a quarter each in Germany (25%) and Spain (25%).
- **Gender:** 55% men, 45% women.
- **Products:** almost everyone has 1 product (51%) or 2 products (46%). Only 266 customers have 3 and 60 have 4. These are small groups, so we will be careful with any conclusions about them.

**What we do:** look at the range and spread of the number columns: minimum, maximum, and the values that split customers into quarters.

In [0]:
# Numeric ranges and spread
numeric_cols = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]
display(bronze_churn.select(numeric_cols).summary("count", "min", "25%", "50%", "75%", "max"))

summary,CreditScore,Age,Tenure,Balance,EstimatedSalary
count,10000,10000,10000,10000,10000
min,350,18,0,0.0,11.58
25%,584,32,3,0.0,50972.6
50%,652,37,5,97157.96,100183.05
75%,717,44,7,127638.35,149381.32
max,850,92,10,250898.09,199992.48


**Result.** How to read the table: "25%" means a quarter of the customers are below this value, and "50%" is the customer exactly in the middle (the median).
- **Credit score:** from 350 to 850, with the middle at 652. A normal spread.
- **Age:** from 18 to 92, with the middle at 37. Most customers are between 32 and 44.
- **Tenure:** from 0 to 10 years with the bank, evenly spread.
- **Balance:** the 25% value is 0, which means **at least a quarter of customers have an empty account**. We look at this more closely below.
- **Salary:** from €11.58 to about €200,000. A yearly salary of €11.58 is not realistic, another thing we check below.

### 1.3 Things that look unusual

**What we do:** all the standard checks passed, but while reading the numbers we noticed a few things that are strange for real bank data. Here we count them exactly.

**Why:** none of these break anything, but they affect how we clean the data and how much we trust certain columns. It is better to know now than to be surprised later, or when the professor asks.

In [0]:
oddities = spark.sql(f'''
    SELECT
        COUNT_IF(Surname LIKE '%?%')                     AS surnames_with_question_mark,
        COUNT_IF(Balance = 0)                            AS zero_balance_customers,
        ROUND(AVG(CASE WHEN Balance = 0 THEN 1 ELSE 0 END) * 100, 1) AS zero_balance_pct,
        COUNT_IF(CreditScore = 850)                      AS credit_score_at_850,
        COUNT_IF(EstimatedSalary < 1000)                 AS salary_below_1000,
        ROUND(MIN(EstimatedSalary), 2)                   AS min_salary
    FROM {BRONZE_CHURN}
''')
display(oddities)

surnames_with_question_mark,zero_balance_customers,zero_balance_pct,credit_score_at_850,salary_below_1000,min_salary
92,3617,36.2,233,59,11.58


**Result:**

| What we found | How many | What it means | What we will do |
|---|---|---|---|
| Surnames containing "?" | 92 | Letters were lost before we received the file | Nothing, since surnames are not used |
| Balance of exactly 0 | 3,617 customers (36%) | A real group of customers, not missing data | Mark them with a "zero balance" flag in GX_02, because this group behaves differently |
| Credit score of exactly 850 | 233 customers | 850 is the top of the scale, so scores were probably capped at 850 | Keep as is, and mention it if asked |
| Yearly salary under €1,000 | 59 customers | Not realistic; salaries look randomly generated | Keep, but do not expect salary to help much in predictions |

## Step 2 — Transaction data (`bank_transactions.csv`)

### 2.1 Loading the file

**What we do:** read the transaction file and save it as the table `bronze_transactions`, the same way as the customer file.

**What is different about this file:**
- **The location column has a comma inside it**, for example `"34.0522 N, -74.006 W"`. Normally a comma means "new column", so we tell Spark to respect the quotes around the value. Without this, every row would be split in the wrong place.
- **We rename the columns** to simple names without spaces (for example `Transaction Amount` becomes `amount`), because Delta tables do not allow spaces or brackets in column names. Only the names change; the values stay exactly the same.
- **Date/time, location and PIN code are kept as text for now.** Turning them into real dates and numbers means interpreting them, and that is a cleaning decision for GX_02. The PIN stays as text so a code like "0123" does not lose its leading zero.

In [0]:
raw_transactions = read_raw_csv(spark, TRANSACTIONS_CSV, TRANSACTIONS_SCHEMA, quote='"', escape='"')
write_bronze(raw_transactions, BRONZE_TRANSACTIONS)

bronze_transactions = spark.table(BRONZE_TRANSACTIONS)
print(f"{BRONZE_TRANSACTIONS}: {bronze_transactions.count():,} rows, {len(bronze_transactions.columns)} columns")
bronze_transactions.printSchema()

workspace.churn.bronze_transactions: 1,000 rows, 17 columns
root
 |-- transaction_id: string (nullable = true)
 |-- sender_account_id: string (nullable = true)
 |-- receiver_account_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- timestamp_raw: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- fraud_flag: boolean (nullable = true)
 |-- geolocation_raw: string (nullable = true)
 |-- device_used: string (nullable = true)
 |-- network_slice_id: string (nullable = true)
 |-- latency_ms: integer (nullable = true)
 |-- slice_bandwidth_mbps: integer (nullable = true)
 |-- pin_code: string (nullable = true)
 |-- _corrupt_record: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



**Result:** **1,000 rows**, exactly as expected. There are 17 columns: the 14 original columns (with their new names) plus the 3 tracking columns.

Each column has the intended type: amounts are decimal numbers, the fraud flag is true/false, and network delay and bandwidth are whole numbers.

**What we do:** look at the first 5 transactions. We hide the PIN code because it is security data.

In [0]:
# Preview without the PIN code (security data is not displayed)
display(bronze_transactions.drop("pin_code", *META_COLS).limit(5))

transaction_id,sender_account_id,receiver_account_id,amount,transaction_type,timestamp_raw,transaction_status,fraud_flag,geolocation_raw,device_used,network_slice_id,latency_ms,slice_bandwidth_mbps,_corrupt_record
TXN9520068950,ACC14994,ACC16656,495.9,Deposit,2025-01-17 10:14:00,Failed,true,"34.0522 N, -74.006 W",Desktop,Slice3,10,179,null
TXN9412011085,ACC58958,ACC32826,529.62,Withdrawal,2025-01-17 10:51:00,Success,false,"35.6895 N, -118.2437 W",Mobile,Slice2,11,89,null
TXN4407425052,ACC56321,ACC92481,862.47,Withdrawal,2025-01-17 10:50:00,Failed,false,"48.8566 N, 2.3522 W",Mobile,Slice1,4,53,null
TXN2214150284,ACC48650,ACC76457,1129.88,Transfer,2025-01-17 10:56:00,Success,true,"34.0522 N, -74.006 W",Mobile,Slice3,10,127,null
TXN4247571145,ACC60921,ACC11419,933.24,Deposit,2025-01-17 10:25:00,Success,true,"55.7558 N, 37.6173 W",Mobile,Slice3,20,191,null


**Result:** the values look as expected: transaction IDs like `TXN…`, account IDs like `ACC…`, amounts in the hundreds, and types like Deposit, Withdrawal and Transfer.

But look at the location column: `48.8566 N, 2.3522 W`. 48.86° North is Paris, but Paris is at 2.35° **East**, not West. We check this properly in section 2.3.

### 2.2 Quality checks

**What we do:** run exactly the same quality function as for the customers. That is the benefit of writing it once. It checks for missing values, duplicates, sensible ranges (amount, network delay, bandwidth) and expected categories.

In [0]:
transactions_quality = quality_report(
    bronze_transactions,
    dataset="transactions",
    expected_rows=EXPECTED_ROWS["transactions"],
    key_col="transaction_id",
    ranges={
        "amount": (0, 1_000_000),
        "latency_ms": (0, 1_000),
        "slice_bandwidth_mbps": (0, 10_000),
    },
    allowed_values={
        "transaction_type": ["Deposit", "Withdrawal", "Transfer"],
        "transaction_status": ["Success", "Failed"],
        "device_used": ["Mobile", "Desktop"],
        "network_slice_id": ["Slice1", "Slice2", "Slice3"],
    },
    rate_cols=["fraud_flag"],
)

display(transactions_quality.groupBy("status").count().orderBy("status"))
display(transactions_quality.filter("status != 'PASS'"))

status,count
INFO,1
PASS,36


dataset,check,column,observed,expected,status
transactions,positive_rate,fraud_flag,0.4810,-,INFO


**Result:** **36 checks passed, 0 warnings, 0 failures.** Technically, the file is clean.

The one INFO line is surprising, though: **48.1% of transactions are flagged as fraud.** In real banking, fraud is well under 1% of transactions. Almost half being fraud suggests the data was generated randomly, so we must be careful before drawing any conclusions from it.

**What we do:** count each category in the transaction file.

In [0]:
display(category_counts(bronze_transactions, ["transaction_type", "transaction_status", "device_used", "network_slice_id", "fraud_flag"]))

column,value,n,pct
device_used,Mobile,521,52.1
device_used,Desktop,479,47.9
fraud_flag,false,519,51.9
fraud_flag,true,481,48.1
network_slice_id,Slice2,340,34.0
network_slice_id,Slice3,337,33.7
network_slice_id,Slice1,323,32.3
transaction_status,Failed,513,51.3
transaction_status,Success,487,48.7
transaction_type,Transfer,374,37.4


**Result:** every category is split almost evenly:
- Mobile 52% vs Desktop 48%
- Failed 51% vs Success 49%
- Transfer 37%, Deposit 32%, Withdrawal 31%
- About a third for each network slice
- Fraud 48% vs not fraud 52%

Real data is rarely this balanced. This is another sign that the file is artificial.

### 2.3 Does the data make sense? (checks with Spark SQL)

**What we do:** the standard checks told us the file is *technically* fine. They cannot tell us whether it is *realistic* or *useful*. So we answer four questions using Spark SQL (the same query language used in databases):

1. **How much time does the data cover?** Patterns need time to appear.
2. **Are the locations real places?**
3. **Do accounts send money to each other more than once?** We need repeated connections to build a network (graph) in Phase 6.
4. **Can we link transactions to the customers in `churn.csv`?** If we can, transactions could help predict churn.

In [0]:
# 1. Time coverage
display(spark.sql(f'''
    SELECT
        MIN(TO_TIMESTAMP(timestamp_raw))                         AS first_transaction,
        MAX(TO_TIMESTAMP(timestamp_raw))                         AS last_transaction,
        COUNT_IF(TO_TIMESTAMP(timestamp_raw) IS NULL)            AS unparseable_timestamps,
        COUNT(DISTINCT DATE_TRUNC('MINUTE', TO_TIMESTAMP(timestamp_raw))) AS distinct_minutes
    FROM {BRONZE_TRANSACTIONS}
'''))

first_transaction,last_transaction,unparseable_timestamps,distinct_minutes
2025-01-17T10:01:00.000Z,2025-01-17T11:00:00.000Z,0,60


**Result — time:** all 1,000 transactions happened within **one single hour** (17 January 2025, from 10:01 to 11:00), spread over 60 different minutes. Every date/time was readable.

One hour is far too short to see customer behaviour or trends.

**Next question:** are the locations real?

In [0]:
# 2. Geolocation: format, number of distinct latitudes/longitudes, hemisphere letters
display(spark.sql(f'''
    WITH parsed AS (
        SELECT
            geolocation_raw,
            REGEXP_EXTRACT(geolocation_raw, '^(-?[0-9.]+) [NS]', 1)  AS lat_text,
            REGEXP_EXTRACT(geolocation_raw, ', (-?[0-9.]+) [EW]$', 1) AS lon_text,
            RIGHT(geolocation_raw, 1)                                AS lon_letter
        FROM {BRONZE_TRANSACTIONS}
    )
    SELECT
        COUNT(*)                                   AS total_rows,
        COUNT_IF(lat_text = '' OR lon_text = '')   AS badly_formatted,
        COUNT(DISTINCT lat_text)                   AS distinct_latitudes,
        COUNT(DISTINCT lon_text)                   AS distinct_longitudes,
        COUNT(DISTINCT geolocation_raw)            AS distinct_points,
        COUNT_IF(lon_letter = 'W')                 AS labelled_west,
        COUNT_IF(lon_text LIKE '-%')               AS negative_longitude
    FROM parsed
'''))

total_rows,badly_formatted,distinct_latitudes,distinct_longitudes,distinct_points,labelled_west,negative_longitude
1000,0,6,6,36,1000,310


**Result — locations:** there are only **6 different latitudes and 6 different longitudes**, combined in all 36 possible ways. The numbers belong to six real cities (New York, Los Angeles, London, Paris, Moscow, Tokyo), but they are mixed up at random, for example Paris's latitude paired with Tokyo's longitude. On top of that, **all 1,000 points are labelled "W" (West)**, even when the number says otherwise.

**Conclusion:** the locations are not real, so we will not use them for any location analysis.

**Next question:** do accounts transact more than once?

In [0]:
# 3. Account network: how often does each account appear (as sender or receiver)?
display(spark.sql(f'''
    WITH appearances AS (
        SELECT sender_account_id AS account_id FROM {BRONZE_TRANSACTIONS}
        UNION ALL
        SELECT receiver_account_id FROM {BRONZE_TRANSACTIONS}
    ),
    per_account AS (
        SELECT account_id, COUNT(*) AS n_transactions FROM appearances GROUP BY account_id
    )
    SELECT n_transactions, COUNT(*) AS n_accounts
    FROM per_account
    GROUP BY n_transactions
    ORDER BY n_transactions
'''))

n_transactions,n_accounts
1,1943
2,27
3,1


**Result — network:** the 1,000 transactions involve **1,971 different accounts**. **1,943 accounts appear only once**, 27 appear twice, and just 1 appears three times. Almost nobody sends money to anyone twice.

**Conclusion:** a network built from this data will be mostly small, disconnected pairs of accounts. We will still analyse it in Phase 6, but the main network analysis will be built from customer similarity instead.

**Next question:** can we link transactions to customers?

In [0]:
# 4. Join key: transaction accounts look like 'ACC12345'; customer IDs are 8-digit numbers.
display(spark.sql(f'''
    WITH tx_accounts AS (
        SELECT sender_account_id AS account_id FROM {BRONZE_TRANSACTIONS}
        UNION
        SELECT receiver_account_id FROM {BRONZE_TRANSACTIONS}
    )
    SELECT
        COUNT(*)                                    AS distinct_tx_accounts,
        COUNT_IF(t.account_id RLIKE '^ACC[0-9]+$')  AS accounts_in_ACC_format,
        COUNT(c.CustomerId)                         AS numeric_part_matches_a_customer
    FROM tx_accounts t
    LEFT JOIN (SELECT DISTINCT CustomerId FROM {BRONZE_CHURN}) c
        ON CAST(REGEXP_EXTRACT(t.account_id, '([0-9]+)', 1) AS BIGINT) = c.CustomerId
'''))

distinct_tx_accounts,accounts_in_ACC_format,numeric_part_matches_a_customer
1971,1971,0


**Result — link to customers:** all 1,971 transaction accounts have IDs like `ACC12345`, while customers have 8-digit IDs like `15634602`. **None of them match.**

**Conclusion:** the two files cannot be joined, so transactions cannot help predict churn. We treat them as a separate, smaller analysis.

## Step 3 — Saving all check results

**What we do:** combine the quality results of both files into one table, `quality_report`, together with the time the checks were run.

**Why:** the results are saved permanently. Anyone in the team (or the professor) can look at them later without re-running this notebook. If the data ever changes, we can compare the new results with these.

In [0]:
full_report = (
    churn_quality.unionByName(transactions_quality)
    .withColumn("_checked_at", F.current_timestamp())
)
write_bronze(full_report, QUALITY_REPORT)

display(
    spark.table(QUALITY_REPORT)
    .groupBy("dataset")
    .pivot("status", ["PASS", "WARN", "FAIL", "INFO"])
    .count()
    .fillna(0)
)

dataset,PASS,WARN,FAIL,INFO
churn,34,0,0,3
transactions,36,0,0,1


**Result:**
- **Customers:** 34 PASS, 0 WARN, 0 FAIL, 3 INFO
- **Transactions:** 36 PASS, 0 WARN, 0 FAIL, 1 INFO

Both files passed every check.

**What we do:** a final safety check. If any check had failed, this cell would stop the notebook with an error, so nobody accidentally builds on broken data.

In [0]:
# Gate: later notebooks assume bronze is structurally sound. Stop here if any hard check failed.
failed = spark.table(QUALITY_REPORT).filter("status = 'FAIL'")
n_failed = failed.count()
if n_failed > 0:
    display(failed)
    raise ValueError(f"{n_failed} quality check(s) failed - fix ingestion before continuing.")
print("All hard checks passed - bronze tables are ready for GX_02.")

All hard checks passed - bronze tables are ready for GX_02.


**Result:** no check failed. The tables are safe to use in the next notebook.

## Summary

### What we did
- Loaded both files with fixed column types, kept any broken rows (there were none), and added tracking columns.
- Saved them as tables: `bronze_churn` (10,000 customers) and `bronze_transactions` (1,000 transactions).
- Ran **74 automatic checks** on the two files. Every one passed.
- Investigated the things that looked unusual, and saved all results in `quality_report`.

### What we learned

**Customer data: clean and ready to use.**
- No missing values, no duplicates, no impossible values.
- **20.4% of customers left the bank.** This is what we will predict. Because the groups are uneven (1 in 5 leaves), we will need careful evaluation in Phase 4.
- **36% of customers have a balance of 0.** This is a real group, and we will mark it explicitly.
- Salary looks randomly generated and credit scores look capped at 850. These are not errors, but we will not rely heavily on those columns.

**Transaction data: technically clean, but not realistic.**
- It covers only one hour, almost half is flagged as fraud, and the locations are mixed up at random.
- Almost no account appears twice, so the network is very sparse.
- It cannot be linked to the customers.
- We will use it only for a small separate analysis and be honest about its limits in the presentation.

**Bottom line:** the customer data is a solid base for the whole project. The transaction data is weak, and we know exactly why.

---

## Next: GX_02 — Data engineering

Now that we know the data can be trusted, the next notebook prepares it for analysis and modelling. It follows the same layer idea:

**1. Silver (cleaned data)**
- **Customers:** keep only the useful columns. Row number and surname are dropped. Customer ID is kept only to identify customers, never as a model input.
- **Transactions:** turn the date/time text into real dates, turn the locations into numbers, and remove the PIN code for privacy.

**2. Gold (ready for analysis)** — add new columns that describe customers better, based on what we saw here:
- **Zero balance** (yes/no), since 36% of customers have an empty account
- **Age group** (18–30, 31–40, 41–50, 51–60, 60+), since leaving the bank changes a lot with age
- **Number of products** grouped as 1 / 2 / 3+
- **Tenure group** (0–2, 3–6, 7–10 years)
- **Engagement score**, combining active membership, credit card and having more than one product
- **Balance-to-salary ratio**

**3. Model preparation steps** — models only understand numbers, so country and gender are turned into numbers and all values are put on a similar scale. These steps are learned only from the training data, so our results stay honest.

**4. Showing why Spark** — we copy the customer data 100 times (about 1 million rows) and run the same steps again, to show the client that the code scales. This is the main reason to use Spark.

After GX_02, every later notebook (analysis, model, network) reads the gold table, so the whole team works from the same prepared data.